# Predicting salary

In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import pickle


In [2]:
# Reading CSV file
df = pd.read_csv('Churn_Modelling.csv')
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
# Preprocessing the data

## Removing un-necessary column
df = df.drop(['RowNumber','CustomerId','Surname'],axis=1)

In [4]:
# Making pipeline for encoding

reg_preprocessor = ColumnTransformer(
    transformers=[
        ('binary',OrdinalEncoder(categories=[['Female','Male']]),['Gender']),
        ('ohe',OneHotEncoder(sparse_output=False),['Geography']),
        ('scale',StandardScaler(),['CreditScore','Age','Tenure','Balance','NumOfProducts','HasCrCard','IsActiveMember','Exited'])
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)


reg_preprocessor = ColumnTransformer([
    ('binary', OrdinalEncoder(categories=[['Female','Male']]), ['Gender']),
    ('ohe', OneHotEncoder(sparse_output=False), ['Geography']),
    ('scale', StandardScaler(), [
        'CreditScore',
        'Age',
        'Tenure',
        'Balance',
        'NumOfProducts'
    ])
], remainder='passthrough')

In [5]:
# Spliting Data

X = df.drop(columns=['EstimatedSalary'])
y = df['EstimatedSalary']

In [6]:
# Spliting the data into train & test
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.20,random_state=42)

pred_scaled = model.predict(X_test_processed)
pred = y_scaler.inverse_transform(pred_scaled)

In [7]:
# applying preprocessor
X_train_transformed = reg_preprocessor.fit_transform(X_train) # now, it is in pure array, just numbers no dataframe
X_test_transformed = reg_preprocessor.transform(X_test)

 

In [8]:
with open("reg_preprocessor.pkl","wb") as f:
    pickle.dump(reg_preprocessor,f)

In [9]:
y_scaler = StandardScaler()

y_train_transformed = y_scaler.fit_transform(
    y_train.values.reshape(-1,1)  #expects 2d array i.e '.reshape(-1,1)'
)

y_test_transformed = y_scaler.transform(
    y_test.values.reshape(-1,1)
)

In [10]:
print(y_train_transformed.shape)
print(y_test_transformed.shape)

(8000, 1)
(2000, 1)


## ANN with Regression problem

In [11]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

2026-06-22 21:05:16.676349: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-22 21:05:16.681496: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-22 21:05:16.753488: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-22 21:05:16.753549: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-22 21:05:16.755353: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

In [12]:
model = Sequential([
    Dense(64, activation='relu',input_shape=(X_train_transformed.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1) # Output layer for regression
])

In [13]:
## Compile the model 
model.compile(optimizer='adam',loss='mse',metrics=['mae'])

# Display the summary
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 


Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [14]:
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

# Setup TensorBoard
log_dir = "regressionlogs/fit/"+ datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)


In [15]:
## Set up early Stopping

early_stopping_callback = EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [16]:
# Train the model 
history = model.fit(
    X_train_transformed,y_train_transformed,
    validation_data=(X_test_transformed,y_test_transformed),
    epochs = 100,
    batch_size = 32,
    callbacks = [early_stopping_callback,tensorboard_callback]
)

Epoch 1/100
250/250 [==============================] - 3s 9ms/step - loss: 1.0209 - mae: 0.8688 - val_loss: 1.0144 - val_mae: 0.8692
Epoch 2/100
250/250 [==============================] - 2s 7ms/step - loss: 0.9995 - mae: 0.8604 - val_loss: 1.0137 - val_mae: 0.8705
Epoch 3/100
250/250 [==============================] - 2s 7ms/step - loss: 0.9916 - mae: 0.8581 - val_loss: 1.0228 - val_mae: 0.8738
Epoch 4/100
250/250 [==============================] - 2s 7ms/step - loss: 0.9863 - mae: 0.8549 - val_loss: 1.0212 - val_mae: 0.8743
Epoch 5/100
250/250 [==============================] - 1s 5ms/step - loss: 0.9820 - mae: 0.8534 - val_loss: 1.0201 - val_mae: 0.8730
Epoch 6/100
250/250 [==============================] - 1s 5ms/step - loss: 0.9794 - mae: 0.8516 - val_loss: 1.0204 - val_mae: 0.8727
Epoch 7/100
250/250 [==============================] - 1s 5ms/step - loss: 0.9741 - mae: 0.8488 - val_loss: 1.0246 - val_mae: 0.8738
Epoch 8/100
250/250 [==============================] - 1s 5ms/step - 

250/250 [==============================] - 1s 5ms/step - loss: 49603.4141 - mae: 49603.4141 - val_loss: 49880.8867 - val_mae: 49880.8867

when y not transformed & everything scaled

Epoch 79/100
250/250 [==============================] - 1s 5ms/step - loss: 49599.8477 - mae: 49599.8477 - val_loss: 49898.9609 - val_mae: 49898.9609

when binary are not scaled.

Epoch 11/100
250/250 [==============================] - 1s 5ms/step - loss: 0.8459 - mae: 0.8459 - val_loss: 0.8810 - val_mae: 0.8810

when binary are not scaled and the target is scaled

In [17]:
## After prediction, inverse-transform
y_pred_scaled = model.predict(X_test_transformed)

y_pred = y_scaler.inverse_transform(y_pred_scaled)


from sklearn.metrics import r2_score
r2 = r2_score(y_test, y_pred)
print(r2)

63/63 [==============================] - 0s 3ms/step
-0.01573372991645483


In [18]:
test_loss,test_mae=model.evaluate(X_test_transformed,y_test_transformed)
print(f"Test MAE : {test_mae}")

63/63 [==============================] - 0s 4ms/step - loss: 1.0137 - mae: 0.8705
Test MAE : 0.8704912066459656
